# Multi-format RAG ingestion pipeline
Loads **PDF, DOCX, HTML, CSV and TXT/MD** files from a folder, chunks them, embeds them, and upserts everything into Pinecone.

In [ ]:
%pip install -q pypdf docx2txt beautifulsoup4 lxml \
    langchain langchain-community langchain-text-splitters langchain-huggingface \
    sentence-transformers pinecone groq python-dotenv pandas tqdm

## Imports

In [ ]:
import os
from pathlib import Path
from typing import Callable, Dict, List

from dotenv import load_dotenv
from tqdm.auto import tqdm

from langchain_core.documents import Document as LangchainDocument
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import (
    TextLoader,
    PyPDFLoader,
    Docx2txtLoader,
    BSHTMLLoader,
    CSVLoader,
)

from pinecone import Pinecone
from groq import Groq

load_dotenv()

## Configuration

In [ ]:
DATA_DIR = "./data"  # folder containing the raw files to ingest (any mix of types below)

EMBEDDING_MODEL_NAME = "thenlper/gte-small"

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
PINECONE_INDEX_NAME = "lab-rag-index"
PINECONE_NAMESPACE = "ns1"

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GROQ_MODEL = os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile")

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 100
UPSERT_BATCH_SIZE = 100

## Multi-format document loaders
Each loader returns a list of `langchain_core.documents.Document`. Add new extensions to `EXTENSION_LOADERS` to support more formats.

In [ ]:
def load_text(path: str) -> List[LangchainDocument]:
    return TextLoader(path, encoding="utf-8").load()


EXTENSION_LOADERS: Dict[str, Callable[[str], List[LangchainDocument]]] = {
    ".txt": load_text,
    ".md": load_text,
    ".pdf": lambda path: PyPDFLoader(path).load(),
    ".docx": lambda path: Docx2txtLoader(path).load(),
    ".html": lambda path: BSHTMLLoader(path).load(),
    ".htm": lambda path: BSHTMLLoader(path).load(),
    ".csv": lambda path: CSVLoader(path).load(),
}


def load_any_file(path: str) -> List[LangchainDocument]:
    ext = Path(path).suffix.lower()
    loader = EXTENSION_LOADERS.get(ext)
    if loader is None:
        print(f"Skipping unsupported file type: {path}")
        return []
    try:
        return loader(path)
    except Exception as exc:
        print(f"Failed to load {path}: {exc}")
        return []


def load_directory(folder: str) -> List[LangchainDocument]:
    all_docs: List[LangchainDocument] = []
    for root, _, files in os.walk(folder):
        for fname in files:
            all_docs.extend(load_any_file(os.path.join(root, fname)))
    return all_docs

## Load every supported file under `DATA_DIR`

In [ ]:
raw_documents = load_directory(DATA_DIR)
print(f"Loaded {len(raw_documents)} source document(s) from {DATA_DIR}")

## Chunking

In [ ]:
MARKDOWN_SEPARATORS = [
    "\n#{1,6} ",
    "```\n",
    "\n\\*\\*\\*+\n",
    "\n---+\n",
    "\n___+\n",
    "\n\n",
    "\n",
    " ",
    "",
]

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    add_start_index=True,
    strip_whitespace=True,
    separators=MARKDOWN_SEPARATORS,
)

docs_processed = text_splitter.split_documents(raw_documents)
print(f"Split into {len(docs_processed)} chunk(s)")

## Embedding model

In [ ]:
embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    encode_kwargs={"normalize_embeddings": True},
)

## Upsert every chunk into Pinecone (batched)

In [ ]:
pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(PINECONE_INDEX_NAME)


def batched(items, size):
    for i in range(0, len(items), size):
        yield items[i:i + size]


for batch_start, batch in enumerate(tqdm(list(batched(docs_processed, UPSERT_BATCH_SIZE)))):
    vectors = [
        {
            "id": f"vec{batch_start * UPSERT_BATCH_SIZE + i}",
            "values": embedding_model.embed_query(doc.page_content),
            "metadata": {
                "text": doc.page_content,
                "source": doc.metadata.get("source", "unknown"),
            },
        }
        for i, doc in enumerate(batch)
    ]
    index.upsert(vectors=vectors, namespace=PINECONE_NAMESPACE)

print("Ingestion complete.")

## Quick retrieval + Groq generation smoke test

In [ ]:
groq_client = Groq(api_key=GROQ_API_KEY)

prompt_template = """You are a helpful assistant that answers medical questions using only the provided context.
If the context doesn't contain the answer, say "I don't know".

Context:
{context}

Question: {question}
"""


def ask(question: str, top_k: int = 3) -> str:
    query_vector = embedding_model.embed_query(question)
    results = index.query(
        namespace=PINECONE_NAMESPACE,
        vector=query_vector,
        top_k=top_k,
        include_metadata=True,
    )
    context = "\n".join(match["metadata"]["text"] for match in results["matches"])
    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": prompt_template.format(context=context, question=question)}],
    )
    return response.choices[0].message.content


print(ask("What are the symptoms of diabetes?"))